<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **8. Wi-Fi Single Link Throughput**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)

  Remark: If you cannot run the simulations, use the illustrations that are already available.
  
</div>




### 8.1 Introduction

The motivation for this notebook is to analyze the basics of the WLAN IEEE 802.11 medium access protocol and understand the WLAN IEEE 802.11 system performance in scenarios with one single transmitting device, and no competition.


In order to prepare it might be helpful to read through the WLAN chapters of the course material. Further, the first five pages of the paper on 802.11e found in the background readings should be helpful ([click here for access](https://gitlab.ethz.ch/wireless/BackgroundReading/-/blob/main/802.11/802.11e.pdf)):

>S. Mangold, Sunghyun Choi, G. R. Hiertz, O. Klein and B. Walke, "Analysis of IEEE 802.11e for QoS support in wireless LANs," in IEEE Wireless Communications, vol. 10, no. 6, pp. 40-50, Dec. 2003.

In [ ]:
import platform

classpath_jemula = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar"
classpath_jemula802 = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar:jemula/lib/commons-io-2.14.0.jar"

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")



#### 8.1.1 Clone the Jemula802 Simulator 

Jemula802 is an open-source tool for simulating IEEE 802.11 wireless networks, released under the BSD license. It supports network simulations with statistical traffic generators and includes simple TCP/IP stacks, directional antennas, and a mobility model. Wireless signal path loss is represented using a radio propagation model, while modulation and coding schemes are approximated with heuristic interference models, similar to those used in link budget analysis. Developed in Java, Jemula802 is built on an event-driven architecture. Its default connectivity model calculates packet errors based on signal, noise, and interference levels, taking into account packet structures as well as coding and modulation schemes. Simulation scenarios are configured through XML files, where nearly every parameter of the wireless protocols and system can be customized. Notably, the MAC layer parameters are independent and can be set individually for each simulated radio station, enabling fine-grained control over experiments.

In [ ]:
%%bash
set -e
if [ -d "jemula802/.git" ]; then
  echo "Updating existing repo jemula802 ..."
  git -C jemula802 pull --ff-only
else
  echo "Cloning repo jemula802 ..."
  git clone https://bitbucket.org/lfield/jemula802.git
fi
set -e
if [ -d "jemula/.git" ]; then
  echo "Updating existing repo jemula ..."
  git -C jemula pull --ff-only
else
  echo "Cloning repo jemula ..."
  git clone https://bitbucket.org/lfield/jemula.git
fi

#### 8.1.2 Compile jemula802 and jemula Java Code

In [ ]:
!javac -g:none -classpath $classpath_jemula -d "jemula/bin" jemula/src/plot/*.java jemula/src/kernel/*.java jemula/src/statistics/*.java
!echo Compilation of jemula done.

!javac -g:none  -classpath $classpath_jemula802 -d "jemula802/bin" jemula802/src/emulator/*.java jemula802/src/layer0_medium/*.java jemula802/src/layer1_802Phy/*.java \
 jemula802/src/layer1_80211Phy/*.java jemula802/src/layer2_802Mac/*.java jemula802/src/layer2_80211Mac/*.java \
 jemula802/src/layer2_802Algorithms/*.java jemula802/src/layer3_network/*.java jemula802/src/layer5_application/*.java \
 jemula802/src/layer4_transport/*.java jemula802/src/station/*.java jemula802/src/visualization/*.java \
 jemula802/src/util/*.java jemula802/src/gui/*.java
!echo Compilation of jemula802 done.

import os

ROOT_DIR = os.getcwd()
!echo Root directory is $ROOT_DIR

### 8.2 Simulations

#### 8.2.1 Scenario "MobComp_assignment03_1000byte.xml"
Note the name of the XML Scenario file below. This file, [ETH_Course/MobComp_assignment03_1000byte.xml](jemula802/scenarios/ETH_Course/MobComp_assignment03_1000byte.xml), defines what will be simulated: A single link, one station sending to another station, with a constant stream of data. Frame payloads are between 0 and 1000 bytes long.

>This simulation models a point-to-point Wi-Fi (IEEE 802.11) link between two stations: one fixed at (0,0) and one mobile at (10,10). The network runs on channel 6 (2.437 GHz) with omnidirectional antennas, 0 dBm transmit power, and a 50 m coverage range. Routing and multichannel switching are disabled, so it is a single-hop link. The MAC layer uses 802.11e QoS with EDCA parameters, and the PHY supports OFDM modes up to 54 Mb/s (64QAM 3/4) with modeled bit error probabilities. The simulation runs for 50 seconds, collecting throughput, delay, and offered load statistics. Traffic is generated at the mobile station as multiple parallel UDP flows (1000-byte packets, mean 0.75 Mb/s each, best-effort class), all directed to the fixed station. These flows start one after another from 2 s up to 37 s, progressively increasing the traffic load over time. The goal is to evaluate how a Wi-Fi link performs under increasing offered traffic, in terms of throughput and delay, using realistic MAC/PHY behavior.


In [ ]:
os.chdir(ROOT_DIR)
EXAMPLE_SCENARIO="jemula802/scenarios/ETH_Course/MobComp_assignment03_1000byte.xml"
!echo Running example scenario $EXAMPLE_SCENARIO
!java -Xmx500m -classpath $classpath_jemula802 emulator.JE802Starter $EXAMPLE_SCENARIO

#### 8.2.2 Scenario "MobComp_assignment03_120byte.xml"
Note the name of the XML Scenario file below. This file, [ETH_Course/MobComp_assignment03_120byte.xml](jemula802/scenarios/ETH_Course/MobComp_assignment03_120byte.xml), defines what will be simulated: a single link, one station sending to another station, with a constant stream of data. Frame payloads are between 0 and only 120 bytes long, which is the only difference to the previous simulation scenario, above.

>This second scenario is identical in setup (two stations, same PHY/MAC settings, same UDP load pattern, same duration) except that the **traffic packet size is reduced from 1000 bytes to 120 bytes** and the results folder name changes accordingly. This means the offered bit-rate per flow (0.75 Mb/s) is the same, but many more small packets are sent. The effect is that MAC/PHY overhead becomes much larger per bit, fragmentation no longer applies (since 120 B < 512 B threshold), and the channel sees a much higher packet rate. As a result, the simulation will produce **lower efficiency, more contention, and greater delay/jitter per packet** compared to the 1000-byte case, while modeling how packet size impacts Wi-Fi performance.


In [ ]:
os.chdir(ROOT_DIR)
EXAMPLE_SCENARIO="jemula802/scenarios/ETH_Course/MobComp_assignment03_120byte.xml"
!echo Running example scenario $EXAMPLE_SCENARIO
!java -Xmx500m -classpath $classpath_jemula802 emulator.JE802Starter $EXAMPLE_SCENARIO

### 8.3 Evaluation

The simulation results and a copy of the XML Scenario file are stored in 
"[results_MobComp_A03_1000byte](results_MobComp_A03_1000byte)" and "[results_MobComp_A03_120byte](results_MobComp_A03_120byte)"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

os.chdir(ROOT_DIR)

In [ ]:
def plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay):
    fig, ax = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax.cla()
    ax.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax.axis([None, None, None, None])
    ax.set_facecolor((.2, 0.2, 0.3))
    plt.xlim(0, np.max(result_total_offer.iloc[:, 0]) / 1000 * 1.05)  # Set x-axis limits
    plt.xlabel('Simulated Time [s]')
    plt.ylabel('Simulation Results, Mb/s & ms')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks

    plt.plot(result_total_offer.iloc[:, 0] / 1000, result_total_offer.iloc[:, 6], '-', color='g', linewidth=4, label='Offer [Mb/s]')
    plt.plot(result_total_thrp.iloc[:, 0] / 1000, result_total_thrp.iloc[:, 6], '-', color='w', linewidth=1, label='Throughput [Mb/s]')
    plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 3], '-', color='r', linewidth=1, label='Avg. Delay [ms]')
    #plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 5], '-', color='r', linewidth=1)
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

def plot_delay_histogram(aBinSize_ms_string, aHistogram_string):
    fig, ax_distr = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax_distr.cla()
    ax_distr.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax_distr.axis([None, None, None, None])
    ax_distr.set_facecolor((.2, 0.2, 0.3))
    theHistogram = [float(x) for x in aHistogram_string.split()]
    theHistogram = [x/sum(theHistogram) for x in theHistogram]
    theNumOfBins = len(theHistogram)
    theBinSize_ms = float(aBinSize_ms_string)
    theX = [i * theBinSize_ms + theBinSize_ms/2 for i in range(theNumOfBins)] 
    theCumSum = 1-np.cumsum(theHistogram)
    theCumSum = np.insert(theCumSum, 0, 1)
    plt.step([0] + theX, [0] + theHistogram, '-', color='r', linewidth=2, label='PDF')
    plt.step([0] + theX, theCumSum, '-', color='m', linewidth=2, label='CDF')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks
    plt.xlim(0, theNumOfBins * theBinSize_ms * 1.05)  # Set x-axis limits
    plt.xlabel('x [ms]')
    plt.ylabel('PDF: P(delay=x) & CCDF: P(delay>=x)')
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

#### 8.3.1 Plot the Results for 1000 Bytes Maximum Frame Body Size

In [ ]:
if os.path.isdir("results_MobComp_A03_1000byte"):
    os.chdir("results_MobComp_A03_1000byte")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

#### 8.3.2 Plot the Results for 120 Bytes Maximum Frame Body Size

In [ ]:
if os.path.isdir("results_MobComp_A03_120byte"):
    os.chdir("results_MobComp_A03_120byte")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

----
### 8.4 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**What is the purpose of `dot11FragmentationThreshold`? See page 19 in the course document** "***05 Mobile Computing - Emerging Technologies - IEEE 802.11 Wi-Fi - MAC Layer"***.  Answer:

(add your answer here)

**What is the purpose of `dot11RTSThreshold`? See page 20 in the course document** "***05 Mobile Computing - Emerging Technologies - IEEE 802.11 Wi-Fi - MAC Layer"*** Answer:

(add your answer here)


**You see four plots. Explain the first throughput plot. What is the maximum throughput, and how do you explain the value?** Answer:

(add your answer here)

**What is a typical transmission delay of a packet (second plot)? Is this significantly influenced by time of flight (speed of light)? How do you explain the delay graph?** Answer:

(add your answer here)

**What is the concept of a payload? What are advantages and disadvantages of a large payload (example: 1000 bytes)? What are the dis-/advantages of smaller payloads (example: 120 bytes)?** Answer:

(add your answer here)

**Explain the throughput differences between the two scenarios.** Answer:

(add your answer here)

**Explain the delay differences between the two scenarios.** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)